# Model 1 on a base that never saw ORD — `sagawa/CompoundT5`, 57k (Kaggle GPU)

**Why.** Every Model 1 result so far starts from `sagawa/ReactionT5v2-retrosynthesis`, which
was pretrained on ~1.5M ORD reactions (snapshot 2024-05-06, 80/10/10 split). `data/v2_ord_eval_targets.json`
is sampled from that same `Open-Reaction-Database/ord-data`, and the ReactionT5 authors only
removed overlaps with the USPTO-50k and C-N test sets — this project's ORD test did not exist
then. So the ORD numbers in `RESULTS.md` are an upper bound of unknown tightness.

`sagawa/CompoundT5` is that same checkpoint one step earlier: per the paper, *"ReactionT5 is
initialized with the weights of CompoundT5 and further pre-trained on reaction data"*. CompoundT5
is span-MLM over 24M ZINC20 **molecules** — no reactions at all. Identical architecture
(`T5ForConditionalGeneration`, d_model 768, d_ff 2048, 12+12 layers, 12 heads); only the vocab
differs, 221 vs 268. With this base the ORD test is uncontaminated by construction, so no
train/test overlap analysis is needed.

**Vocabulary repair is mandatory here.** ZINC20 is a library of single drug-like molecules, so
CompoundT5's tokenizer has no `.` (the fragment separator, 158,963 occurrences in this project's
data) and none of the letters that spell out metals and counterions. 61,067 of 115,200 SMILES
strings across the ORD/USPTO eval targets and the 57k train split tokenize with at least one
`<unk>`; without the repair the model cannot emit a multi-fragment reactant set and
`exact_match` is 0 by construction. `train_reactant_model_ord.py` now calls
`retro_eval.tokenizer_coverage.ensure_full_char_coverage`, which adds the 30 missing characters
(vocab 221 -> 251) and resizes the embeddings with `mean_resizing=False`. On ReactionT5 the same
call is inert (verified: of 114,000 strings only the 40 that already carried an `<unk>` tokenize
differently afterwards), so earlier variants keep their recipe.

**Data:** 57,000 ORD reactions, the exact split variant 2 used — dataset
`kuzmenkooleh/retro-planner-reactants-57k-uspto` (the mixed-in USPTO file it also carries is not
used here). Same `--no-augment`, 3 epochs, `torchrun --nproc_per_node=2` launch as notebooks 03,
04 and 09, so this run differs from variant 2 in the base checkpoint and the learning rate only.

**Before running:** Settings panel -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All
(Commit)**, not a Draft Session. The notebook clones the repo from GitHub, so any script change
must be pushed first.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

## Reference row: untuned CompoundT5

Scores the base checkpoint with its **original** 221-token vocabulary (no repair — that only
happens inside training), which is what "the base checkpoint before fine-tuning" means for the
`RESULTS.md` table. Expect 0.0% on every metric: a span-MLM model has never produced a reactant
set, and without a `.` token it cannot express one. `valid_top1` in the summary is the number
that says *why* — the same role `json_valid_rate_top1` = 0.0% plays for Model 2's untuned
`t5-small` row.

~10 minutes for both test sets.

In [ ]:
for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model sagawa/CompoundT5 \
        --num-beams 10 --device cuda --batch-size 16 \
        --output "/kaggle/working/compoundt5_baseline_{tag}_topk.json"
    print(tag, "baseline done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_baseline_{tag}_topk.json"))
    print("===", tag, "baseline ===")
    print(json.dumps(data["summary"], indent=2))
    print("sample candidates:", data["records"][0]["candidates"][:3])

## Fine-tuning

**On the learning rate.** Variant 2's `5e-5` was tuned for a base that already knew how to emit
a reactant set; CompoundT5 has to learn the task *and* 30 freshly initialized embedding rows,
one of which is the most frequent character in the targets (`.`). The direct precedent in this
project is Model 2: plain `t5-small`, also a general base, needed `5e-4` — ten times Model 1's
rate. So this probe runs at `5e-4` to answer "does CompoundT5 learn retrosynthesis at all".

Set `learning_rate = 5e-5` for the follow-up run: that one is the clean ablation against variant
2 (identical data and hyperparameters, base checkpoint the only difference) and is worth doing
once this probe shows the task is learnable at all.

57,000 examples at an effective batch of 32 is ~5,344 steps; at the 0.97 steps/s measured on the
150k DDP run that is ~92 min, plus ~15 min for the two evaluations that follow.

In [ ]:
import glob

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4
output_dir = "/kaggle/working/model1_compoundt5_57k"
time_budget_minutes = 200  # ~92 min training + headroom

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The repair must have fired: 221 -> 251. If this line is absent the run trained on
# <unk>-corrupted targets and its numbers are meaningless (the Model 2 failure mode,
# where teacher-forced eval_loss looked healthy while generation was 0% valid).
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

# Counted via the tokenizer object and config.vocab_size, not tokenizer.json's
# model.vocab: this is a Unigram tokenizer, so characters added at training time land
# in a separate `added_tokens` list and model.vocab alone undercounts badly (a saved
# ReactionT5 checkpoint reads 121 there against 151 added_tokens).
saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

probe = "CC(=O)O.[K+].[Si](C)C"
ids = saved_tokenizer(probe, add_special_tokens=False)["input_ids"]
print("unk in probe:", saved_tokenizer.unk_token_id in ids, "|", repr(saved_tokenizer.decode(ids)))

In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/compoundt5_57k_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_57k_{tag}_topk.json"))
    print("===", tag, "fine-tuned ===")
    print(json.dumps(data["summary"], indent=2))
    print("sample candidates:", data["records"][0]["candidates"][:3])